# Control: equal supervision at every episode length

**Requires a GPU.** Produced `data/curves_fixedquery.csv`.

**The objection this answers.** A longer episode also carries more supervised
query rows, so at an equal number of gradient steps it sees more targets. Any
advantage of long episodes could therefore be supervision rather than length.

**The control.** The same grid, with the query count held at **128 for every
episode length**. Supervision is then equal by construction and only the context
length varies. The effect survives: +2.73 points between the longest and shortest
episode at contexts of 1024 rows and above, 13 datasets out of 13, p = 0.0002.

*Provenance only - the analysis notebook reads the committed CSV.*

---


## Fixed-query control — isolating the training context length

**Before running:** attach the output of the previous run as an input. This
notebook re-reads its base curve and verifies it, exactly as the previous one
did.

**The objection this answers.** In the previous grid, enlarging the episode also
enlarged the number of supervised query rows: 26, then 128, then 410. At an equal
number of gradient steps, the longest episode therefore saw roughly sixteen times
more supervised targets than the shortest. Any advantage it showed could be
supervision rather than length, and the two were confounded.

**The control.** The same grid, with the query count held at **128 for every
episode length**. Supervision is then equal by construction and only the context
length varies. The import is filtered accordingly: arms from the previous run
with 26 or 410 queries are rejected, since re-using them would reintroduce the
confound this notebook exists to remove.

If the effect survives here, it is the length. If it disappears, it was the
supervision, and that must be reported plainly.

In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','tabpfn==9.0.0',
                'openml>=0.15,<0.17','scikit-learn>=1.5,<1.8','scipy','matplotlib','nbformat>=5'],check=True)
print('Dependencies installed. Restart the kernel if Kaggle asks, then run all.')

In [ ]:

import os
os.environ['TABPFN_NO_BROWSER']='1'
if not os.environ.get('TABPFN_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception as exc:
        raise RuntimeError('Create and enable the Kaggle secret TABPFN_TOKEN (Prior Labs API key).') from exc
    if not token: raise RuntimeError('The TABPFN_TOKEN secret is empty.')
    os.environ['TABPFN_TOKEN']=token.strip(); del token
print('Credentials loaded in memory, key never printed.')

## 1. Configuration

In [ ]:

from pathlib import Path

SEED = 42
SUITE_ID = 457
ALL_TASK_IDS = [363621,363626,363619,363623,363682,363684,
                363613,363618,363624,363627,363629,363632,
                363671,363674,363676,363679,363681,363689,
                363691,363694,363696,363700,363706,363712]
N_SHARDS, SHARD_ID = 1, 0

# ---- axis 1: the inference context (identical to the previous run, or nothing
#      can be reused) --------------------------------------------------------
CONTEXT_SIZES     = (32,64,128,256,512,1024,2048,None)
REF_CONTEXT       = 1024
SMALL_CONTEXT_MAX = 256
REPEATS_SMALL     = 3

# ---- axis 2: the length of a training episode -----------------------------
# (context, query); the query share stays at 0.2 throughout.
# CONTROL: the number of query rows is FIXED. At an equal number of steps the three
# arms therefore see exactly the same number of supervised targets, and the only
# remaining variable is the length of the training context.
FT_QUERY_FIXE      = 128
EPISODE_FIXED      = ((102,128),(512,128))                 # 230 and 640 rows
EPISODE_CANDIDATES = ((1638,128),(1229,128),(819,128))     # 1766, 1357, 947 rows
MIN_ELIGIBLE       = 10      # an episode length below this is not worth the compute
ELIG_SLACK         = 1.25    # train >= 1.25 x episode: headroom for stratification

FT_MILESTONES = (25,50,200)
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP = 1e-5, 0.01, 1.0

# ---- axis 3: how aggressive the optimiser is ------------------------------
# Real-TabPFN (arXiv 2507.03971) obtains GAINS with 3e-7, zero decay and an
# L2-SP anchor to the original weights. That is the most predictable objection to
# a negative result measured at 1e-5: "you are not measuring fine-tuning, you are
# measuring too large a step". This phase answers it.
# The step budget is raised so that the weight DISPLACEMENT is of the same order:
# the comparison is at matched displacement, not at matched steps.
RUN_LR_PHASE   = False   # already measured in the previous run
LR_LOW         = 1e-6        # 10x smaller; 3e-7 over 200 steps would move nothing
LR_LOW_WD      = 0.0         # decay toward zero is replaced by the anchor
L2SP_ALPHA     = 0.003       # Real-TabPFN's value
LR_MILESTONES  = (250,1000)
LR_EPISODE     = (512,128)   # mid-sized episode, already covered elsewhere
LR_PHASE_TASKS = 4

# ---- evaluation ----------------------------------------------------------
REPORT_EVAL_CAP = 1024
PRED_BATCHES    = (64,16,4,1)

# ---- importing the previous run -------------------------------------------
IMPORT_PREVIOUS   = True
IMPORT_TOLERANCE  = 5e-3   # beyond this the whole import is rejected, not patched
IMPORT_CHECK_N    = 3      # datasets re-measured to verify the import

# ---- time budget ---------------------------------------------------------
TIME_BUDGET_HOURS = 8.0
CURVE_SECONDS_GUESS = 120.0   # cost of one full curve, refined after the first measured one

# OpenML returns 503 regularly; retry rather than lose the run.
OPENML_RETRIES, OPENML_BACKOFF = 6, 10

SKIP_ON_ERROR = True
MIN_TARGETS = 12
MAX_DATASET_ROWS, MAX_FEATURES = 50_000, 110
MIN_FREE_GIB = 3.0
LOCAL_TRAIN_CAP = REF_CONTEXT

RUN_TAG = 'requetes_fixes_v1'
ON_KAGGLE = Path('/kaggle/working').exists()
ROOT    = (Path('/kaggle/working') if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG
SCRATCH = (Path('/kaggle/temp')   if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG
ROOT.mkdir(parents=True,exist_ok=True); SCRATCH.mkdir(parents=True,exist_ok=True)

SHARD_IDS = ALL_TASK_IDS[SHARD_ID::N_SHARDS]
assert len(ALL_TASK_IDS)==len(set(ALL_TASK_IDS))==24
assert REF_CONTEXT in CONTEXT_SIZES and None in CONTEXT_SIZES
assert list(FT_MILESTONES)==sorted(set(FT_MILESTONES))
assert all(a>b for (a,_),(b,_) in zip(EPISODE_CANDIDATES,EPISODE_CANDIDATES[1:])), \
       'candidates must be ordered from largest to smallest'
assert EPISODE_CANDIDATES[-1][0]>EPISODE_FIXED[-1][0], 'the 3rd size must exceed the two already measured'
assert all(q==FT_QUERY_FIXE for _,q in EPISODE_FIXED+EPISODE_CANDIDATES), \
       'the control requires an identical query count everywhere'

def arm_name(ep_rows,steps,regime='standard'):
    return (f'ep{int(ep_rows)}_{int(steps)}' if regime=='standard'
            else f'{regime}_ep{int(ep_rows)}_{int(steps)}')

print('Results:',ROOT)
print('Contexts  :',CONTEXT_SIZES)
print('Training contexts:',[a for a,_ in EPISODE_FIXED],'+ a 3rd chosen by probe')
print(f'Queries per episode: {FT_QUERY_FIXE} - IDENTICAL everywhere, this is the control')
print('Total supervision per arm:',{s:s*FT_QUERY_FIXE for s in FT_MILESTONES},'queries')
print('FT budgets:',FT_MILESTONES,'steps | time budget',TIME_BUDGET_HOURS,'h')

## 2. Imports and utilities

In [ ]:

import re, gc, json, time, random, shutil, hashlib, inspect, traceback, importlib.metadata
import numpy as np, pandas as pd, torch, openml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from scipy.stats import spearmanr
from IPython.display import display

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from tabpfn.base import get_embeddings as tabpfn_get_embeddings
from tabpfn.model_loading import save_tabpfn_model
from tabpfn.finetuning.finetuned_classifier import FinetunedTabPFNClassifier
from tabpfn.finetuning.finetuned_base import _parameters_unused_by_task
from tabpfn.finetuning.data_util import get_preprocessed_dataset_chunks, meta_dataset_collator
from tabpfn.finetuning._torch_compat import GradScaler, autocast, sdpa_kernel_context
from tabpfn.architectures.interface import PerformanceOptions

assert importlib.metadata.version('tabpfn')=='9.0.0', 'tabpfn != 9.0.0: restart the kernel after pip.'
assert torch.cuda.is_available(), 'Enable the GPU (Settings -> Accelerator -> GPU).'
assert hasattr(ModelVersion,'V3_5')
assert 'shuffle' in inspect.signature(get_preprocessed_dataset_chunks).parameters
DEVICE='cuda:0'; GIB=2**30
print('GPU :',torch.cuda.get_device_name(0),'/',round(torch.cuda.get_device_properties(0).total_memory/GIB,2),'Gio')

def clean_gpu():
    gc.collect(); torch.cuda.empty_cache()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sha(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    tmp.replace(p)

def write_csv(path,frame):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(p)

def check_disk(label,where=None,extra=0.0):
    free=shutil.disk_usage(where or SCRATCH).free/GIB
    if free < MIN_FREE_GIB+extra:
        raise RuntimeError(f'{label}: {free:.1f} GiB free. Export the CSVs, then re-run a smaller shard.')

def choose_indices(y,n,seed):
    # Deterministic stratified subsample, at least 2 rows per class.
    y=np.asarray(y); n=min(int(n),len(y))
    classes,counts=np.unique(y,return_counts=True)
    if len(classes)!=2 or min(counts)<2: raise ValueError('Two classes with >= 2 rows required')
    if n==len(y): return np.arange(len(y))
    if n<4: raise ValueError('Sample too small')
    k1=max(2,min(int(counts[1]),int(round(n*counts[1]/len(y))))); k0=n-k1
    if k0>counts[0]: k0=int(counts[0]); k1=n-k0
    if k0<2: k0=2; k1=n-k0
    if k1>counts[1] or k0>counts[0] or min(k0,k1)<2: raise ValueError('Stratification impossible')
    rng=np.random.default_rng(int(seed))
    a=rng.choice(np.flatnonzero(y==classes[0]),k0,replace=False)
    b=rng.choice(np.flatnonzero(y==classes[1]),k1,replace=False)
    return np.sort(np.r_[a,b])

def scores(y,p):
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any(): raise ValueError('Invalid probabilities')
    if len(np.unique(y))!=2: raise ValueError('Metric undefined on a single-class split')
    return dict(auc=float(roc_auc_score(y,p)),
                logloss=float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),
                ap=float(average_precision_score(y,p)))

def is_oom(e):
    return isinstance(e,torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()

FAILURES=[]
def record_failure(phase,tid,name,exc,**extra):
    # A failure is dropped from the run, never hidden: it lands in failures.csv and on screen.
    row=dict(phase=phase,task_id=tid,name=name,error=type(exc).__name__,
             message=str(exc)[:400],**extra)
    FAILURES.append(row)
    p=ROOT/'failures.csv'
    tmp=p.with_name(p.name+'.tmp'); pd.DataFrame(FAILURES).to_csv(tmp,index=False); tmp.replace(p)
    print(f'  !! DROPPED [{phase}] {name} ({tid}) - {type(exc).__name__}: {str(exc)[:160]}',flush=True)
    if not SKIP_ON_ERROR: raise exc
    return row

def openml_retry(fn,*a,**kw):
    # OpenML returns 503 regularly, or cuts out mid-download. We retry with
    # growing backoff rather than lose a run of several hours.
    last=None
    for k in range(OPENML_RETRIES):
        try: return fn(*a,**kw)
        except Exception as exc:
            last=exc
            if k==OPENML_RETRIES-1: break
            wait=min(OPENML_BACKOFF*(2**k),120)
            print(f'    OpenML unavailable ({type(exc).__name__}) - retrying '
                  f'dans {wait}s [{k+1}/{OPENML_RETRIES-1}]',flush=True)
            time.sleep(wait)
    raise last

set_seed(SEED)
print('Utilities ready.')

## 3. Metadata and splits

**Rigorously identical to the previous run**: same task ids, same seed, same
splits. That is the condition for imported measurements to be comparable with new
ones, and the import check verifies it rather than assuming it.

In [ ]:

openml.config.set_root_cache_directory(str(SCRATCH/'openml_cache'))
check_disk('OpenML metadata')
# Suite membership: a useful check, but not at the cost of the run.
# The 24 task ids are frozen and were already validated in earlier runs.
try:
    suite=openml_retry(openml.study.get_suite,SUITE_ID)
    allowed=set(map(int,suite.tasks))
    assert set(ALL_TASK_IDS)<=allowed
    print('Suite 457 verified.')
except Exception as exc:
    print(f'Suite 457 not verifiable ({type(exc).__name__}) - continuing with the '
          '24 frozen task ids.',flush=True)

def clean_description(s):
    s=re.sub(r'<[^>]+>',' ',str(s or ''))
    s=re.sub(r'https?://\S+',' ',s)
    return re.sub(r'\s+',' ',s).strip()[:2500]

META={}; TASKS={}; TEXT={}; seen=set()
for tid in list(ALL_TASK_IDS):
    try:
        t=openml_retry(openml.tasks.get_task,tid,download_data=False,download_qualities=False)
        ds=openml_retry(t.get_dataset,download_data=False)
        did=int(t.dataset_id)
        if did in seen: raise ValueError(f'{tid}: duplicate OpenML dataset_id')
        txt=f'{ds.name}. Prediction target {t.target_name}. '+clean_description(ds.description)
        if len(txt)<20: raise ValueError(f'{tid}: description too short for the text baseline')
        seen.add(did)
        META[tid]={'task_id':tid,'dataset_id':did,'name':str(ds.name),'target':str(t.target_name),
                   'description':txt,'url':f'https://www.openml.org/t/{tid}'}
        TASKS[tid]=t; TEXT[tid]=txt
        print(tid,META[tid]['name'])
    except Exception as exc:
        record_failure('metadata',tid,f'task_{tid}',exc)
ALL_TASK_IDS=[t for t in ALL_TASK_IDS if t in TASKS]
if len(ALL_TASK_IDS)<MIN_TARGETS:
    raise RuntimeError(f'only {len(ALL_TASK_IDS)} tasks: see failures.csv.')
print(f'{len(ALL_TASK_IDS)}/24 OpenML tasks available.')
write_json(ROOT/'catalog_meta.json',META)

In [ ]:

def encode_task(frames):
    # frames[0] = train: the only split the preprocessing is fitted on.
    frames=[f.copy().reset_index(drop=True) for f in frames]
    good=[c for c in frames[0].columns
          if frames[0][c].notna().sum()>0 and frames[0][c].nunique(dropna=True)>1]
    if not good: raise ValueError('No usable feature')
    nums=[c for c in good if pd.api.types.is_numeric_dtype(frames[0][c])]
    cats=[c for c in good if c not in nums]
    parts=[]
    if nums: parts.append(('numeric',SimpleImputer(strategy='median'),nums))
    if cats: parts.append(('categorical',OrdinalEncoder(handle_unknown='use_encoded_value',
                            unknown_value=-1,encoded_missing_value=-1,dtype=np.float32),cats))
    prep=ColumnTransformer(parts,remainder='drop',sparse_threshold=0)
    clean=[]
    for f in frames:
        f=f[good].copy()
        for c in nums: f[c]=pd.to_numeric(f[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        for c in cats: f[c]=f[c].astype('string').fillna('__MISSING__').astype(str)
        clean.append(f)
    out=[]
    for j,f in enumerate(clean):
        x=prep.fit_transform(f) if j==0 else prep.transform(f)
        x=np.asarray(x,dtype=np.float32)
        x=np.clip(np.nan_to_num(x,nan=0,posinf=1e4,neginf=-1e4),-1e4,1e4)
        if not np.isfinite(x).all(): raise ValueError('Non-finite X values')
        out.append(np.ascontiguousarray(x))
    return out,{'n_features':len(good),'numeric':len(nums),'categorical':len(cats),
                'feature_name_sha256':hashlib.sha256(json.dumps([str(c) for c in good]).encode()).hexdigest()}

def get_bundle(tid):
    task=TASKS[tid]; ds=openml_retry(task.get_dataset)
    X,y,_,_=openml_retry(ds.get_data,target=task.target_name)
    if not isinstance(X,pd.DataFrame) or isinstance(y,pd.DataFrame):
        raise TypeError('Unexpected OpenML X/y format')
    outer,sealed=task.get_train_test_split_indices(fold=0,repeat=0,sample=0)
    outer=np.asarray(outer); sealed=np.asarray(sealed)
    assert np.intersect1d(outer,sealed).size==0 and len(outer)+len(sealed)==len(X)
    label=LabelEncoder().fit(pd.Series(y).iloc[outer].astype(str))
    if len(label.classes_)!=2: raise ValueError(f'{tid}: task is not binary')
    y_outer=label.transform(pd.Series(y).iloc[outer].astype(str))
    if min(np.bincount(y_outer))<20: raise ValueError(f'{tid}: minority class too rare')
    i_fit,i_report=train_test_split(outer,test_size=0.20,stratify=y_outer,random_state=SEED)
    y_fit=label.transform(pd.Series(y).iloc[i_fit].astype(str))
    i_train,i_val=train_test_split(i_fit,test_size=0.30,stratify=y_fit,random_state=SEED+1)
    y_val_tmp=label.transform(pd.Series(y).iloc[i_val].astype(str))
    i_score,i_select=train_test_split(i_val,test_size=0.50,stratify=y_val_tmp,random_state=SEED+2)
    for a,b in [(i_train,i_score),(i_train,i_select),(i_train,i_report),
                (i_score,i_select),(i_score,i_report),(i_select,i_report)]:
        assert np.intersect1d(a,b).size==0, f'{tid}: leakage between splits'
    enc=lambda idx: label.transform(pd.Series(y).iloc[idx].astype(str)).astype(np.int64)
    ytr,ysc,ysl,yrp=enc(i_train),enc(i_score),enc(i_select),enc(i_report)
    for nm,z in [('train',ytr),('Vscore',ysc),('Vselect',ysl),('report',yrp)]:
        if min(np.bincount(z))<4: raise ValueError(f'{tid}: class too rare in {nm}')
    (Xtr,Xsc,Xsl,Xrp),feat=encode_task([X.iloc[i_train],X.iloc[i_score],X.iloc[i_select],X.iloc[i_report]])
    if len(X)>MAX_DATASET_ROWS or Xtr.shape[1]>MAX_FEATURES:
        raise ValueError(f'{tid}: outside the caps ({len(X)} rows, {Xtr.shape[1]} features)')
    ctx=choose_indices(ytr,min(LOCAL_TRAIN_CAP,len(ytr)),SEED+tid)
    b=dict(Xtrain=Xtr,ytrain=ytr,Xscore=Xsc,yscore=ysc,Xselect=Xsl,yselect=ysl,
           Xreport=Xrp,yreport=yrp,Xcontext=Xtr[ctx],ycontext=ytr[ctx],
           context_idx=ctx,cat_indices=[],
           meta={**META[tid],'n_train':len(ytr),'n_score':len(ysc),'n_select':len(ysl),
                 'n_report':len(yrp),'n_context':len(ctx),'features':feat,
                 'sealed_outer_test_rows':len(sealed),
                 'split_sha256':hashlib.sha256(np.sort(i_train).astype('<i8').tobytes()).hexdigest()})
    return b

def bundle_cache_paths(tid):
    # A previous run attached as an input saves depending on OpenML again.
    cands=[ROOT/'bundles'/f'{tid}.npz']
    inp=Path('/kaggle/input')
    if inp.exists(): cands+=sorted(inp.glob(f'**/bundles/{tid}.npz'))
    return cands

def load_bundle(tid):
    for p in bundle_cache_paths(tid):
        if not p.exists(): continue
        try:
            with np.load(p,allow_pickle=False) as a:
                b={k:a[k] for k in a.files if k!='meta_json'}
                b['meta']=json.loads(str(a['meta_json'].item()))
            b['cat_indices']=[]
            return b
        except Exception as exc:
            print(f'  unreadable cache {p} ({type(exc).__name__})',flush=True)
    return None

def save_bundle(tid,b):
    d=ROOT/'bundles'; d.mkdir(parents=True,exist_ok=True)
    p=d/f'{tid}.npz'; tmp=p.with_name(p.name+'.tmp')
    arrays={k:v for k,v in b.items() if isinstance(v,np.ndarray)}
    with tmp.open('wb') as f:
        np.savez_compressed(f,meta_json=np.array(json.dumps(b['meta'],default=str)),**arrays)
    tmp.replace(p)

BUNDLES={}; rows=[]; from_cache=0
for tid in ALL_TASK_IDS:
    try:
        cached=load_bundle(tid)
        if cached is not None:
            BUNDLES[tid]=cached; from_cache+=1
        else:
            check_disk(f'loading {tid}')
            BUNDLES[tid]=get_bundle(tid); save_bundle(tid,BUNDLES[tid])
        m=BUNDLES[tid]['meta']
        rows.append({k:m[k] for k in
            ['task_id','name','n_train','n_score','n_select','n_report','n_context']}
            |{'n_features':m['features']['n_features']})
        print(f"{m['name'][:42]:<42} | train {m['n_train']:>5} Vscore {m['n_score']:>4} "
              f"Vselect {m['n_select']:>4} report {m['n_report']:>5} | context {m['n_context']:>4} "
              f"| variables {m['features']['n_features']:>3}")
    except Exception as exc:
        record_failure('bundle',tid,META[tid]['name'],exc)
        clean_gpu()

print(f'{from_cache}/{len(ALL_TASK_IDS)} datasets re-read from a local or attached cache.')
ACTIVE_IDS=[t for t in ALL_TASK_IDS if t in BUNDLES]
if len(ACTIVE_IDS)<MIN_TARGETS:
    raise RuntimeError(f'Only {len(ACTIVE_IDS)} datasets loaded: see failures.csv.')
PARTITIONS=pd.DataFrame(rows); write_csv(ROOT/'partitions.csv',PARTITIONS); display(PARTITIONS)
write_json(ROOT/'bundles_meta.json',{str(t):BUNDLES[t]['meta'] for t in ACTIVE_IDS})
print(f'\n{len(ACTIVE_IDS)}/24 datasets active - each one serves as target and as donor.')

## 4. The episodic engine

Episode length is a parameter, since it is the axis of phase C. No LoRA, no
projection, no parameter subsetting: the whole model is fine-tuned.

The numerical fallback matters on this hardware. A T4 is `sm_75` and has no
bf16, so autocast falls back to fp16 and the backward pass overflows on wide
tables. The loss scale is reduced over eight steps, then the pass is redone in
fp32, then the episode is redrawn. **The `used_amp` flag is load-bearing**: in
fp32 the gradients are already at scale 1, so the caller must use
`optimizer.step()` and never `scaler.step()`, which would divide them by 65536.

In [ ]:

def stratified_pick(y,n,seed):
    y=np.asarray(y); assert len(np.unique(y))==2
    n=min(int(n),len(y))
    if n==len(y): return np.arange(len(y))
    idx,_=train_test_split(np.arange(len(y)),train_size=n,stratify=y,random_state=int(seed))
    if len(np.unique(y[idx]))<2: raise ValueError('Single-class episode')
    return np.sort(idx)

def ft_episode(b,seed,n_context,n_query):
    # Context and query rows are disjoint, both drawn from the training split.
    for k in range(30):
        s=int(seed)+k*100003
        iq=stratified_pick(b['ytrain'],n_query,s)
        pool=np.flatnonzero(~np.isin(np.arange(len(b['ytrain'])),iq))
        if len(pool)<n_context or len(np.unique(b['ytrain'][pool]))<2: continue
        ic=pool[stratified_pick(b['ytrain'][pool],n_context,s+1)]
        assert not (set(ic.tolist()) & set(iq.tolist()))
        return (b['Xtrain'][ic],b['ytrain'][ic],b['Xtrain'][iq],b['ytrain'][iq],s),ic,iq
    raise RuntimeError('Cannot build a disjoint episode')

class EpisodeEngine:
    def __init__(self,checkpoint,cat_indices,seed):
        set_seed(seed)
        kw=dict(device=DEVICE,n_estimators=1,random_state=int(seed),fit_mode='batched',
                differentiable_input=False,memory_saving_mode=True,
                categorical_features_indices=cat_indices,
                inference_config={'ENABLE_GPU_PREPROCESSING':False,'TRANSFORM_TEXT':False,
                                  'TRANSFORM_DATES':False})
        self.est=(TabPFNClassifier.create_default_for_version(ModelVersion.V3_5,**kw)
                  if checkpoint is None else TabPFNClassifier(model_path=str(checkpoint),**kw))
        self.est._initialize_model_variables(); self.est.model_.to(DEVICE)
        for p in _parameters_unused_by_task(self.est.model_,'multiclass'): p.requires_grad=False
        self.wrapper=FinetunedTabPFNClassifier(device=DEVICE,model_version=ModelVersion.V3_5)
        self.wrapper.finetuned_estimator_=self.est
        self.wrapper._ddp_module_=None; self.wrapper._local_n_estimators_=1
        self.performance=PerformanceOptions(force_recompute_layer=True,use_chunkwise_inference=False)
    def params(self):
        return {n:p for n,p in self.est.model_.named_parameters() if p.requires_grad}
    def batch(self,episode):
        xc,yc,xq,yq,seed=episode; k=len(yc)
        assert set(np.unique(yc))=={0,1}
        def fixed_split(X,y,stratify=None): return X[:k],X[k:],y[:k],y[k:]
        ds=get_preprocessed_dataset_chunks(calling_instance=self.est,
            X_raw=np.ascontiguousarray(np.concatenate([xc,xq]),dtype=np.float32),
            y_raw=np.concatenate([yc,yq]),split_fn=fixed_split,max_data_size=None,
            model_type='classifier',equal_split_size=False,data_shuffle_seed=seed,
            preprocessing_random_state=seed,shuffle=False)
        assert len(ds)==1
        return meta_dataset_collator([ds[0]])
    def loss(self,batch):
        self.est.fit_from_preprocessed(batch.X_context,batch.y_context,batch.cat_indices,
                                       batch.configs,performance_options=self.performance)
        self.est.model_.train()
        return self.wrapper._forward_with_loss(batch)
    def save(self,path):
        p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp'); save_tabpfn_model(self.est,tmp); tmp.replace(p)
    def close(self):
        del self.wrapper; del self.est; clean_gpu()

AMP_EVENTS=[]   # log of numerical fallbacks: a measurement in its own right

def _grads_finite(engine):
    return all(p.grad is None or bool(torch.isfinite(p.grad).all())
               for p in engine.params().values())

def backward_checked(engine,batch,optimizer,scaler,tag='',step=None):
    # Returns (loss, used_amp). used_amp=False means gradients are already at scale 1:
    # the caller must use optimizer.step(), never scaler.step().
    scale0=scaler.get_scale()
    for k in range(8):                       # 65536 -> 4, quartering each time
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=True),sdpa_kernel_context(): loss=engine.loss(batch)
        if not torch.isfinite(loss): break   # the problem is in the forward pass, not the backward
        with sdpa_kernel_context(): scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if _grads_finite(engine):
            if k: AMP_EVENTS.append(dict(tag=tag,step=step,mode='amp_rescaled',
                                         reductions=k,scale=scaler.get_scale()))
            return float(loss.detach()),True
        scaler.update(new_scale=max(1.0,scaler.get_scale()/4))
    # fp32 fallback: the fp16 overflow disappears and the step stays exact.
    optimizer.zero_grad(set_to_none=True)
    with autocast(enabled=False),sdpa_kernel_context(): loss=engine.loss(batch)
    if torch.isfinite(loss):
        with sdpa_kernel_context(): loss.backward()
        if _grads_finite(engine):
            scaler.update(new_scale=scale0)
            AMP_EVENTS.append(dict(tag=tag,step=step,mode='fp32'))
            return float(loss.detach()),False
    raise FloatingPointError('Non-finite gradients in AMP and in fp32 alike')

def named_hash(engine):
    h=hashlib.sha256()
    for name,p in sorted(engine.params().items()):
        a=p.detach().cpu().contiguous()
        h.update(name.encode()); h.update(str(tuple(a.shape)).encode())
        h.update(a.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

print('Episodic engine ready.')

## 5. Checkpointed training, and the context curve

Unchanged but for one column: every point now records the episode length that
produced it (`ep_context`, `ep_query`), since that has become the variable of
interest. It is also what allows the ratio context ÷ episode to be computed, the
quantity that should make the three curves collapse onto one.

In [ ]:

BASE=SCRATCH/'base_weights.pth'
if not BASE.exists():
    check_disk('base weights',extra=2.0)
    e=EpisodeEngine(None,[],SEED)
    try: e.save(BASE)
    finally: e.close()
print('Official weights:',BASE,'|',round(BASE.stat().st_size/GIB,2),'Gio')

def predict_ctx(checkpoint,Xctx,yctx,Xq,seed,batch,cat_indices):
    m=TabPFNClassifier(model_path=str(checkpoint),device=DEVICE,n_estimators=1,
        fit_mode='low_memory',memory_saving_mode=True,random_state=int(seed),
        categorical_features_indices=cat_indices,ignore_pretraining_limits=True)
    try:
        m.fit(Xctx,yctx); pos=list(m.classes_).index(1); parts=[]
        for st in range(0,len(Xq),batch):
            parts.append(np.asarray(m.predict_proba(Xq[st:st+batch])[:,pos],dtype=float))
        p=np.concatenate(parts)
        assert len(p)==len(Xq) and np.isfinite(p).all()
        return p
    finally:
        del m; clean_gpu()

def ctx_indices(b,size,seed):
    n=len(b['ytrain']); k=n if size is None else min(int(size),n)
    return choose_indices(b['ytrain'],k,seed)

def eval_slice(b,cap,seed):
    idx=choose_indices(b['yreport'],min(cap,len(b['yreport'])),seed)
    return b['Xreport'][idx],b['yreport'][idx]

def curve_for(checkpoint,tid,arm,ft_steps,ep_context,ep_query,
              lr=LEARNING_RATE,l2sp=0.0,regime='standard'):
    # One set of weights -> one full curve along the context axis.
    b=BUNDLES[tid]; cat=b['cat_indices']; ntr=len(b['ytrain'])
    Xq,yq=eval_slice(b,REPORT_EVAL_CAP,SEED+900+tid)
    out=[]
    for size in CONTEXT_SIZES:
        if size is not None and size>=ntr:
            continue                      # identical to the full context: do not pay for it twice
        nrep=REPEATS_SMALL if (size is not None and size<=SMALL_CONTEXT_MAX) else 1
        for r in range(nrep):
            try:
                idx=ctx_indices(b,size,SEED+tid+7919*r)
            except ValueError as exc:
                print(f'  {arm} ctx={size}: cannot draw a context ({exc})',flush=True); break
            t0=time.monotonic(); p=None
            for bs in (PRED_BATCH,16,4,1):
                try:
                    p=predict_ctx(checkpoint,b['Xtrain'][idx],b['ytrain'][idx],Xq,SEED,bs,cat)
                    break
                except RuntimeError as exc:
                    if not is_oom(exc): raise
                    clean_gpu()
            if p is None:
                print(f'  {arm} ctx={size}: OOM at {len(idx)} rows, point skipped',flush=True); break
            out.append(dict(run_hash=RUN_HASH,target=tid,name=META[tid]['name'],n_train=ntr,
                            arm=arm,ft_steps=int(ft_steps),
                            ep_context=int(ep_context),ep_query=int(ep_query),
                            episode_rows=int(ep_context+ep_query),
                            lr=float(lr),l2sp=float(l2sp),regime=str(regime),
                            context_size=('full' if size is None else str(int(size))),
                            requested=(ntr if size is None else int(size)),
                            n_context_rows=len(idx),repeat=r,n_report_eval=len(yq),
                            seconds=round(time.monotonic()-t0,1),source='measured here',
                            **scores(yq,p)))
    if not out: raise RuntimeError(f'{arm}: no evaluable context size')
    return out

def displacement(engine,theta0):
    # ||theta - theta_0||, in fp32 on CPU: the only scale on which two learning
    # rates can be compared other than at an equal number of steps.
    s=0.0
    for n,p in engine.params().items():
        if n in theta0: s+=float(((p.detach().float().cpu()-theta0[n])**2).sum())
    return float(np.sqrt(s))

def train_fixed(b,start,path,milestones,seed,tag,n_context,n_query,
                lr=None,wd=None,l2sp=0.0):
    lr=LEARNING_RATE if lr is None else float(lr)
    wd=WEIGHT_DECAY  if wd is None else float(wd)
    path=Path(path); path.mkdir(parents=True,exist_ok=True)
    done=path/'complete.json'; end=max(milestones)
    signature=sha(dict(run=RUN_HASH,start=sha_file(start),steps=sorted(milestones),
                       seed=seed,tag=tag,ep=(n_context,n_query),lr=lr,wd=wd,l2sp=l2sp))
    if done.exists():
        info=json.loads(done.read_text())
        if info['signature']==signature and all(
                Path(c['path']).is_file() and sha_file(c['path'])==c['sha256']
                for c in info['checkpoints'].values()):
            print(tag,': already trained, reused.'); return info
    check_disk(tag,extra=2.0*len(milestones))
    e=EpisodeEngine(start,b['cat_indices'],seed)
    opt=torch.optim.AdamW(list(e.params().values()),lr=lr,weight_decay=wd)
    scaler=GradScaler(); history=[]; ckpts={}; seen=set(); redraws=0
    theta0={n:p.detach().float().cpu().clone() for n,p in e.params().items()}
    anchor={n:t.to(DEVICE) for n,t in theta0.items()} if l2sp>0 else None
    t0=time.monotonic(); torch.cuda.reset_peak_memory_stats(); initial=named_hash(e)
    try:
        for step in range(1,end+1):
            for attempt in range(6):
                episode,ic,iq=ft_episode(b,seed+step*7919+attempt*104729,n_context,n_query)
                batch=e.batch(episode)
                try:
                    loss,used_amp=backward_checked(e,batch,opt,scaler,tag=tag,step=step)
                    break
                except FloatingPointError:
                    redraws+=1
                    AMP_EVENTS.append(dict(tag=tag,step=step,mode='episode_redrawn',attempt=attempt))
                    del batch; clean_gpu()
            else:
                raise FloatingPointError(
                    f'{tag}: 6 consecutive non-finite episodes at step {step}. '
                    'Reproducible instability, not a fluke: report this dataset.')
            seen.update(ic.tolist()); seen.update(iq.tolist())
            if anchor is not None:
                # L2-SP: penalty alpha/2 * ||theta - theta_0||^2, added to the gradient
                # before clipping, as a loss penalty rather than a decoupled decay.
                for n,p in e.params().items():
                    if p.grad is not None and n in anchor:
                        p.grad.add_(p.detach()-anchor[n],alpha=l2sp)
            norm=float(torch.nn.utils.clip_grad_norm_(list(e.params().values()),GRAD_CLIP))
            if used_amp:
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            history.append(dict(step=step,loss=loss,grad_norm_before_clip=norm,
                                amp=used_amp,seconds=time.monotonic()-t0))
            del batch
            if step in milestones:
                ck=path/f'step_{step:04d}.pth'; e.save(ck)
                wh=named_hash(e); dsp=displacement(e,theta0)
                assert wh!=initial,'Weights unchanged after training'
                ckpts[str(step)]=dict(path=str(ck),sha256=sha_file(ck),weights_hash=wh,
                                      displacement=dsp,seconds=time.monotonic()-t0)
                write_csv(path/'training_history.csv',pd.DataFrame(history))
                fp32=int((~pd.DataFrame(history).amp).sum())
                print(f'  {tag} | step {step}/{end} | loss {loss:.5f} | |g| {norm:.3f} '
                      f'| displacement {dsp:.4f}'
                      +(f' | fp32 fallbacks {fp32} | redraws {redraws}' if fp32 or redraws else ''),
                      flush=True)
        info=dict(signature=signature,initial_hash=initial,checkpoints=ckpts,
                  seconds=time.monotonic()-t0,n_train=len(b['ytrain']),lr=lr,wd=wd,l2sp=l2sp,
                  unique_rows_seen=len(seen),episode_rows=n_context+n_query,
                  seconds_per_step=(time.monotonic()-t0)/end,
                  fp32_steps=int((~pd.DataFrame(history).amp).sum()),
                  episode_redraws=redraws,
                  final_loss_mean=float(pd.DataFrame(history).loss.tail(10).mean()),
                  peak_allocated_gib=torch.cuda.max_memory_allocated()/GIB)
        write_json(done,info)
        return info
    finally:
        del opt,scaler,theta0,anchor; e.close()
        if AMP_EVENTS: write_csv(ROOT/'amp_events.csv',pd.DataFrame(AMP_EVENTS))

print('Checkpointed training and context curve ready.')

## 6. Memory preflight and protocol

Identical to the previous run: which query batch fits, whether the full context
of the largest dataset passes, whether the weights move, whether reloading is
exact, and whether a fine-tuned model predicts measurably differently from the
base model.

In [ ]:

attempts=[]; PRED_BATCH=None
for bs in PRED_BATCHES:
    try:
        for tid in ACTIVE_IDS:
            b=BUNDLES[tid]
            predict_ctx(BASE,b['Xcontext'],b['ycontext'],b['Xreport'][:bs],SEED,bs,b['cat_indices'])
        PRED_BATCH=bs; break
    except RuntimeError as exc:
        if not is_oom(exc): raise
        attempts.append(dict(batch=bs,error='CUDA OOM')); clean_gpu()
if PRED_BATCH is None: raise RuntimeError('Even a single query row does not fit in memory.')
print('Query batch size chosen:',PRED_BATCH)

from sklearn.datasets import load_breast_cancer
xp,yp=load_breast_cancer(return_X_y=True)
a_,v_=train_test_split(np.arange(len(yp)),train_size=400,stratify=yp,random_state=SEED)
probe=dict(Xtrain=xp[a_].astype('float32'),ytrain=yp[a_].astype(np.int64),
           Xreport=xp[v_].astype('float32'),yreport=yp[v_].astype(np.int64),
           Xcontext=xp[a_].astype('float32'),ycontext=yp[a_].astype(np.int64),cat_indices=[])
RUN_HASH='preflight'
pf=SCRATCH/'_preflight'
if pf.exists(): shutil.rmtree(pf)
try:
    info=train_fixed(probe,BASE,pf,[1,2],SEED,'preflight',102,26)
    ck=info['checkpoints']['2']['path']
    e=EpisodeEngine(ck,[],SEED)
    try: assert named_hash(e)==info['checkpoints']['2']['weights_hash'],'Incorrect reload'
    finally: e.close()
    q=probe['Xreport'][:16]
    p1=predict_ctx(ck,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    p0=predict_ctx(BASE,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    assert float(np.abs(p0-p1).max())>1e-4,'Fine-tuned model identical to the base model'
    print('PREFLIGHT PASS - max base/tuned gap',round(float(np.abs(p0-p1).max()),5))
finally:
    shutil.rmtree(pf,ignore_errors=True); clean_gpu()

TASK_IDS=[t for t in ACTIVE_IDS if t in SHARD_IDS]

def eligible_for(nc,nq):
    need=int(ELIG_SLACK*(nc+nq))
    return [t for t in TASK_IDS if len(BUNDLES[t]['ytrain'])>=need]

PROTOCOL=dict(seed=SEED,splits='outer_80_20_then_70_15_15_v1',
    context_sizes=[str(c) for c in CONTEXT_SIZES],ref_context=REF_CONTEXT,
    repeats_small=REPEATS_SMALL,small_max=SMALL_CONTEXT_MAX,report_cap=REPORT_EVAL_CAP,
    lr=LEARNING_RATE,wd=WEIGHT_DECAY,clip=GRAD_CLIP)
RUN_HASH=sha(dict(protocol=PROTOCOL,tasks=TASK_IDS,milestones=list(FT_MILESTONES),
                  episodes=[list(e) for e in EPISODE_FIXED]))[:16]
RUN_T0=time.monotonic()
def hours_left(): return TIME_BUDGET_HOURS-(time.monotonic()-RUN_T0)/3600
print('Protocol :',RUN_HASH,'|',len(TASK_IDS),'datasets')
for nc,nq in EPISODE_FIXED:
    print(f'  episodes of {nc+nq:>4} rows: {len(eligible_for(nc,nq)):>2} eligible datasets')

## 7. Importing the previous run

Nothing is taken on trust: the import does not merely read the CSV, it
**verifies**.

1. The attached `manifest.json` must declare exactly the same protocol — same
   seed, same splits, same task ids.
2. A few datasets are **re-measured here** and compared against the imported
   values; beyond a tolerance of 5e-3 the entire import is rejected.
3. Rejection is not patched. Either everything is reused, or everything is
   recomputed.

The point of the import is not to save time for its own sake: it is what makes
the new episode lengths comparable with the old ones on the same datasets.

In [ ]:

CURVE_PATH=ROOT/'curves.csv'
curves=pd.read_csv(CURVE_PATH).to_dict('records') if CURVE_PATH.exists() else []

def collect_previous():
    found=[]
    root=Path('/kaggle/input')
    if not (IMPORT_PREVIOUS and root.exists()): return found
    for man in sorted(root.glob('**/manifest.json')):
        csv=man.parent/'curves.csv'
        if not csv.exists(): continue
        try: cfg=json.loads(man.read_text()).get('config',{})
        except Exception as exc:
            print(f'  {man.parent.name}: unreadable manifest ({type(exc).__name__})'); continue
        bad=[k for k,v in PROTOCOL.items() if cfg.get(k)!=v]
        if bad:
            print(f'  {man.parent.name}: different protocol on {bad} - ignored',flush=True); continue
        mapping={'base':('base',0,0,0)}
        # Format of the 'context x fine-tuning' run: arms ft_* and ftlong_*.
        ep=tuple(cfg.get('ft_episode') or ()); lp=tuple(cfg.get('long_episode') or ())
        if len(ep)==2:
            for st in cfg.get('ft_milestones',[]):
                mapping[f'ft_{st}']=(arm_name(sum(ep),st),int(st),int(ep[0]),int(ep[1]))
        if len(lp)==2:
            for st in cfg.get('long_milestones',[]):
                mapping[f'ftlong_{st}']=(arm_name(sum(lp),st),int(st),int(lp[0]),int(lp[1]))
        # Format of the 'episode x context' run: arms already named ep{rows}_{steps}.
        for e in (cfg.get('episodes') or []):
            if not (isinstance(e,(list,tuple)) and len(e)>=2): continue
            nc,nq=int(e[0]),int(e[1])
            for st in cfg.get('milestones',FT_MILESTONES):
                a=arm_name(nc+nq,int(st)); mapping[a]=(a,int(st),nc,nq)
        df=pd.read_csv(csv)
        df=df[df.arm.isin(mapping)].copy()
        # We only re-import the reference and the arms whose query count matches
        # the control: an arm with 26 or 410 queries would corrupt the comparison.
        df=df[[a=='base' or mapping[a][3]==FT_QUERY_FIXE for a in df.arm]].copy()
        if df.empty:
            print(f'  {man.parent.name}: no recognised arm - ignored'); continue
        df['ft_steps']=[mapping[a][1] for a in df.arm]
        df['ep_context']=[mapping[a][2] for a in df.arm]
        df['ep_query']=[mapping[a][3] for a in df.arm]
        df['episode_rows']=df.ep_context+df.ep_query
        # The imported run had a single optimiser regime; label it as such.
        df['lr']=float(cfg.get('lr',LEARNING_RATE)); df['l2sp']=0.0; df['regime']='standard'
        df['source']=man.parent.name
        df['arm']=[mapping[a][0] for a in df.arm]
        df['run_hash']=RUN_HASH
        found.append(df)
        print(f'  {man.parent.name}: {len(df)} points, arms {sorted(df.arm.unique())}',flush=True)
    return found

def verify_import(df,n_check):
    # Re-measure the base curve on a few datasets and compare against the import.
    got=df[df.arm=='base']
    tids=[t for t in TASK_IDS if t in set(got.target)][:n_check]
    worst=0.0; rows=[]
    for tid in tids:
        fresh=pd.DataFrame(curve_for(BASE,tid,'base',0,0,0))
        a=(fresh.groupby('context_size').logloss.mean())
        b_=(got[got.target==tid].groupby('context_size').logloss.mean())
        common=a.index.intersection(b_.index)
        d=float((a[common]-b_[common]).abs().max()) if len(common) else np.inf
        worst=max(worst,d)
        rows.append(dict(target=tid,name=META[tid]['name'],points=len(common),max_gap=round(d,6)))
        print(f'  {META[tid]["name"][:34]:<34} max log-loss gap: {d:.2e}',flush=True)
    return worst,pd.DataFrame(rows),tids

if not curves:
    print('=== IMPORTING THE PREVIOUS RUN ===',flush=True)
    found=collect_previous()
    if not found:
        print('No reusable output attached: everything will be recomputed.',flush=True)
    else:
        IMP=pd.concat(found,ignore_index=True).drop_duplicates(
            subset=['target','arm','context_size','repeat'],keep='first')
        print(f'\n{len(IMP)} candidate points over {IMP.target.nunique()} datasets. Verifying...',flush=True)
        worst,CHECK,checked=verify_import(IMP,IMPORT_CHECK_N)
        write_csv(ROOT/'import_check.csv',CHECK)
        if worst<=IMPORT_TOLERANCE:
            print(f'IMPORT ACCEPTED - max gap {worst:.2e} <= {IMPORT_TOLERANCE}')
            IMP=IMP[~((IMP.arm=='base')&(IMP.target.isin(checked)))]   # keep the fresh measurements
            curves=pd.concat([IMP,pd.DataFrame(
                [r for t in checked for r in curve_for(BASE,t,'base',0,0,0)])],
                ignore_index=True).to_dict('records')
            write_csv(CURVE_PATH,pd.DataFrame(curves))
        else:
            print(f'IMPORT REJECTED - max gap {worst:.2e} > {IMPORT_TOLERANCE}. '
                  'The protocol differs despite a compatible manifest: everything is recomputed.',flush=True)
            curves=[]
else:
    print(f'{len(curves)} points already in {CURVE_PATH.name}: resuming.',flush=True)

def have(tid,arm):
    return any(int(r['target'])==tid and r['arm']==arm for r in curves)
CUR0=pd.DataFrame(curves)
if len(CUR0):
    print('\nGrid status:')
    print(CUR0.groupby('arm').target.nunique().to_string())
print(f'\nBudget left: {hours_left():.2f} h')

## 8. Preflight: the third episode length

It is not chosen in advance. The probe tries the candidates **from largest to
smallest** and keeps the first that satisfies three conditions:

1. one complete training step passes on the widest dataset;
2. the gradients are finite;
3. enough datasets have a training split large enough to carry it.

This is what turns a hardware constraint into a recorded measurement: the largest
episode this machine can hold is itself a number worth publishing.

In [ ]:

def time_one_step(nc,nq,tid):
    # One real training step, timed, then everything is released.
    b=BUNDLES[tid]; e=EpisodeEngine(BASE,b['cat_indices'],SEED)
    opt=torch.optim.AdamW(list(e.params().values()),lr=LEARNING_RATE)
    scaler=GradScaler()
    try:
        episode,_,_=ft_episode(b,SEED,nc,nq)
        t0=time.monotonic()
        batch=e.batch(episode)
        loss,used=backward_checked(e,batch,opt,scaler,tag='probe',step=0)
        if used: scaler.step(opt); scaler.update()
        else: opt.step()
        torch.cuda.synchronize()
        dt=time.monotonic()-t0
        del batch
        return dt,torch.cuda.max_memory_allocated()/GIB
    finally:
        del opt,scaler; e.close()

CURVE_SECONDS=CURVE_SECONDS_GUESS
if len(CUR0) and 'seconds' in CUR0.columns:
    per=CUR0[CUR0.source=='measured here'].groupby(['target','arm']).seconds.sum()
    if len(per): CURVE_SECONDS=max(30.0,float(per.mean()))   # a 0 would corrupt the whole budget
print(f'Cost of one full curve: {CURVE_SECONDS:.0f} s (measured or estimated).')

EP_LARGE=None; EP_LARGE_IDS=[]; probe_log=[]
for nc,nq in EPISODE_CANDIDATES:
    rows_=nc+nq; elig=eligible_for(nc,nq)
    if len(elig)<MIN_ELIGIBLE:
        probe_log.append(dict(episode=rows_,verdict=f'{len(elig)} eligible datasets < {MIN_ELIGIBLE}'))
        print(f'  {rows_:>4} rows: only {len(elig)} eligible datasets - dropped'); continue
    wide=max(elig,key=lambda t:BUNDLES[t]['Xtrain'].shape[1])
    try:
        dt,peak=time_one_step(nc,nq,wide)
    except RuntimeError as exc:
        if not is_oom(exc): raise
        clean_gpu()
        probe_log.append(dict(episode=rows_,verdict='OOM'))
        print(f'  {rows_:>4} rows: OOM on {META[wide]["name"]} - dropped'); continue
    except FloatingPointError:
        clean_gpu()
        probe_log.append(dict(episode=rows_,verdict='non-finite gradients'))
        print(f'  {rows_:>4} rows: non-finite gradients - dropped'); continue
    finally:
        clean_gpu()
    per_ds=max(FT_MILESTONES)*dt+len(FT_MILESTONES)*CURVE_SECONDS
    budget=max(0.0,hours_left()-0.4)*3600
    kmax=int(budget//per_ds) if per_ds>0 else 0
    print(f'  {rows_:>4} rows: {dt:.2f} s/step, peak {peak:.1f} GiB, {len(elig)} eligible, '
          f'{per_ds/60:.1f} min/dataset -> {kmax} datasets fit in {budget/3600:.2f} h',flush=True)
    probe_log.append(dict(episode=rows_,seconds_per_step=round(dt,3),peak_gib=round(peak,2),
                          eligible=len(elig),minutes_per_dataset=round(per_ds/60,1),fits=kmax))
    if kmax>=MIN_ELIGIBLE:
        EP_LARGE=(nc,nq)
        EP_LARGE_IDS=sorted(elig,key=lambda t:-len(BUNDLES[t]['ytrain']))[:min(kmax,len(elig))]
        break
write_csv(ROOT/'probe_episode.csv',pd.DataFrame(probe_log))

EPISODE_GRID=[(nc,nq,eligible_for(nc,nq)) for nc,nq in EPISODE_FIXED]
if EP_LARGE is not None:
    EPISODE_GRID.append((EP_LARGE[0],EP_LARGE[1],EP_LARGE_IDS))
    print(f'\n3rd size selected: {sum(EP_LARGE)} rows ({EP_LARGE[0]} of context) '
          f'sur {len(EP_LARGE_IDS)} datasets.')
else:
    print('\nNo 3rd size fits: the run will only complete 128 and 640 rows. '
          'That alone is what turns n=4 into n=24.')
write_json(ROOT/'manifest.json',dict(run_hash=RUN_HASH,protocol=PROTOCOL,tasks=TASK_IDS,
    milestones=list(FT_MILESTONES),
    episodes=[[nc,nq,len(ids)] for nc,nq,ids in EPISODE_GRID],
    memory_attempts=attempts,curve_seconds=CURVE_SECONDS))

## 9. Recorded prediction

Printed before any result is computed. The two hypotheses make different
predictions **off the diagonal**; that is where, and only where, they can be told
apart.

In [ ]:

PREREG = '''
CONTROL  The number of query rows is fixed at 128 for every episode size. At an equal
    number of steps the three arms therefore see EXACTLY the same number of supervised
    targets. Any difference between them is attributable to the length of the training
    context alone. This answers the main objection to the previous run: there the query
    share was fixed at 0.2, so making the episode longer also multiplied the supervision
    (26 -> 128 -> 410, a factor of 16 at 200 steps).

A0  THE EFFECT SURVIVES THE CONTROL. At identical supervision, short episodes remain
    harmful beyond 1024 context rows and long episodes remain neutral.
    Prediction: a gap of at least 4 points of relative log-loss between the 102-row arm
    and the 1638-row arm, on targets evaluated beyond 1024 rows.
    If the gap disappears, the effect in the previous run was an artefact of supervision
    and we must say so plainly.

A1  COLLAPSE. The gain does not depend on context and episode separately, but on
    their ratio r = context / episode-context. Test: Spearman(gain, log2 r) within
    of each dataset, all episode lengths pooled, against Spearman(gain,
    log2 context). Prediction: the ratio explains it BETTER, and the three curves plotted
    against r collapse onto one another.

A2  ASYMMETRY. The gain is >= 0 for r <= 1 (deploying shorter than trained) and
    clearly < 0 for r >= 2. Test: Wilcoxon within bands of r.

A3  REFUTING STRICT MATCHING. At a fixed context c, an episode LARGER than
    c is no worse than the episode closest to c. Test: paired comparison
    over datasets eligible at both lengths. If the matched episode wins, then the
    matching hypothesis is the true one and we must say so.

A4  THE CROSSING MOVES. The crossing context (where the gain changes sign)
    grows with episode length. Test: Spearman per dataset, then a sign test.

A5  ROBUSTNESS TO THE OPTIMISER. With a 10x smaller learning rate, zero decay
    and an L2-SP anchor (the Real-TabPFN regime), the SHAPE of the curve is
    unchanged once the two regimes are compared at equal WEIGHT DISPLACEMENT rather
    than at an equal number of steps. Prediction: the damage at large contexts persists.
    If it vanishes, there are two mechanisms and not one, and we must say so.

ALTERNATIVE OUTCOME, to be written up as such if it occurs:
    the curves do not collapse against r, and the matched episode beats the longer
    one. The asymmetry would then be false, matching true, and the practical recipe
    far more expensive: one fine-tuning run per context length.
'''
print(PREREG)
write_json(ROOT/'preregistration.json',dict(run_hash=RUN_HASH,text=PREREG,
    written_before_any_result=True))

## 10. The grid

One episode length at a time, one dataset at a time. Anything already in
`curves.csv` — imported or computed in an earlier session — is never recomputed.
Weights are deleted as soon as a dataset is evaluated.

In [ ]:

FT_DIR=SCRATCH/'ft'; FT_DIR.mkdir(parents=True,exist_ok=True)
TRAIN_PATH=ROOT/'training_summary.csv'
train_rows=pd.read_csv(TRAIN_PATH).to_dict('records') if TRAIN_PATH.exists() else []

def log_training(tid,rows_,nc,nq,info,regime):
    global train_rows
    train_rows=[r for r in train_rows
                if not (int(r['target'])==tid and int(r['episode_rows'])==rows_
                        and r.get('regime',"standard")==regime)]
    for s,c in info['checkpoints'].items():
        train_rows.append(dict(target=tid,name=META[tid]['name'],episode_rows=rows_,
            ep_context=nc,ep_query=nq,regime=regime,lr=info['lr'],l2sp=info['l2sp'],
            steps=int(s),displacement=c['displacement'],seconds=c['seconds'],
            seconds_per_step=info['seconds_per_step'],fp32_steps=info['fp32_steps'],
            episode_redraws=info['episode_redraws'],unique_rows_seen=info['unique_rows_seen'],
            final_loss_mean=info['final_loss_mean'],peak_gib=info['peak_allocated_gib']))
    write_csv(TRAIN_PATH,pd.DataFrame(train_rows))

BASE_IDS=[t for t in TASK_IDS if not have(t,'base')]
if BASE_IDS:
    print(f'=== Base curve missing on {len(BASE_IDS)} datasets ===',flush=True)
    for k,tid in enumerate(BASE_IDS,1):
        if hours_left()<=0: print('Time budget reached.',flush=True); break
        try:
            t0=time.monotonic(); new=curve_for(BASE,tid,'base',0,0,0)
            curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
            print(f'{k}/{len(BASE_IDS)} {META[tid]["name"][:34]:<34} | {time.monotonic()-t0:.0f}s',flush=True)
        except Exception as exc:
            record_failure('base',tid,META[tid]['name'],exc); clean_gpu()

for nc,nq,ids in EPISODE_GRID:
    rows_=nc+nq
    todo=[t for t in ids if not all(have(t,arm_name(rows_,s)) for s in FT_MILESTONES)]
    print(f'\n=== EPISODES OF {rows_} ROWS - {len(ids)} eligible, {len(todo)} to do ===',flush=True)
    for k,tid in enumerate(todo,1):
        if hours_left()<=0.15:
            print('Time budget reached: moving on to analysing what has been measured.',flush=True); break
        b=BUNDLES[tid]; path=FT_DIR/f'ep{rows_}_{tid}'
        print(f'\n----- {k}/{len(todo)} {META[tid]["name"]} | train {len(b["ytrain"])} '
              f'| episode {rows_} rows | {hours_left():.2f} h left -----',flush=True)
        try:
            check_disk(f'fine-tuning {tid}',extra=2.0*len(FT_MILESTONES))
            info=train_fixed(b,BASE,path,FT_MILESTONES,SEED+tid,
                             f'ep{rows_} {META[tid]["name"][:20]}',nc,nq)
            log_training(tid,rows_,nc,nq,info,'standard')
            for s in FT_MILESTONES:
                arm=arm_name(rows_,s)
                if have(tid,arm): continue
                new=curve_for(info['checkpoints'][str(s)]['path'],tid,arm,s,nc,nq)
                curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
                base={r['context_size']:r['logloss'] for r in curves
                      if int(r['target'])==tid and r['arm']=='base' and int(r['repeat'])==0}
                g={r['context_size']:round(base.get(r['context_size'],np.nan)-r['logloss'],4)
                   for r in new if r['repeat']==0}
                print(f'  {arm} | gain by context: {g}',flush=True)
        except Exception as exc:
            record_failure(f'ep{rows_}',tid,META[tid]['name'],exc); clean_gpu()
        finally:
            shutil.rmtree(path,ignore_errors=True); clean_gpu()

CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\nGrid: {CUR.target.nunique()} datasets, {CUR.arm.nunique()} arms, {len(CUR)} points. '
      f'{len(FAILURES)} drop(s). Budget left: {hours_left():.2f} h.')

## 10b. The optimiser axis — answering Real-TabPFN

[Real-TabPFN](https://arxiv.org/abs/2507.03971) obtains **gains** by continuing
the pre-training of TabPFN on real data, with a much smaller learning rate, zero
weight decay and an L2-SP anchor toward the original weights. That is the most
predictable objection to a negative result measured at 1e-5: *you are not
measuring fine-tuning, you are measuring too large a step.*

This phase answers it, and the answer only counts if the comparison is fair. Two
learning rates cannot be compared at an equal number of steps — the smaller one
would simply have moved less. They are compared at **equal weight displacement**,
‖θ − θ₀‖ computed in fp32 on CPU, which is the only scale on which the two
regimes are commensurable. The step budget of the low-rate arm is raised until
its displacement matches.

If the shape of the curve survives, the negative result is not an artefact of the
optimiser. If it disappears, there are two mechanisms rather than one, and that
must be reported as such.

In [ ]:

nc,nq=LR_EPISODE; rows_=nc+nq
# The comparison is at matched weight displacement, so we prefer datasets whose
# standard arm was TRAINED here, the only ones with a measured displacement.
# Arms imported from the previous run have none: they predate this measurement,
# and could not serve as a reference.
trained_std=set()
if Path(TRAIN_PATH).exists():
    _ts=pd.read_csv(TRAIN_PATH)
    if len(_ts) and 'displacement' in _ts.columns:
        _m=_ts[(_ts.regime=='standard')&(_ts.episode_rows==rows_)&_ts.displacement.notna()]
        trained_std={int(t) for t in _m.target}
_pool=[t for t in eligible_for(nc,nq) if t in trained_std] or eligible_for(nc,nq)
LR_IDS=(sorted(_pool,key=lambda t:-len(BUNDLES[t]['ytrain']))[:LR_PHASE_TASKS]
        if RUN_LR_PHASE else [])
if RUN_LR_PHASE and not trained_std:
    print('WARNING: no standard arm trained here at this episode length. '
          'The matched-displacement comparison will be impossible.',flush=True)
if LR_IDS and hours_left()>0.5:
    print(f'\n=== OPTIMISER AXIS - lr={LR_LOW}, L2-SP alpha={L2SP_ALPHA}, '
          f'milestones {LR_MILESTONES}, episodes of {rows_} rows ===',flush=True)
    for k,tid in enumerate(LR_IDS,1):
        arms=[arm_name(rows_,s,'lrlo') for s in LR_MILESTONES]
        if all(have(tid,a) for a in arms):
            print(f'{k}/{len(LR_IDS)} {META[tid]["name"]}: already done.'); continue
        if hours_left()<=0.15:
            print('Time budget reached: optimiser axis stopped.',flush=True); break
        b=BUNDLES[tid]; path=FT_DIR/f'lrlo_{tid}'
        print(f'\n----- {k}/{len(LR_IDS)} {META[tid]["name"]} | train {len(b["ytrain"])} '
              f'| {hours_left():.2f} h left -----',flush=True)
        try:
            check_disk(f'low lr {tid}',extra=2.0*len(LR_MILESTONES))
            info=train_fixed(b,BASE,path,LR_MILESTONES,SEED+tid,
                             f'lrlo {META[tid]["name"][:20]}',nc,nq,
                             lr=LR_LOW,wd=LR_LOW_WD,l2sp=L2SP_ALPHA)
            log_training(tid,rows_,nc,nq,info,'lrlo')
            for s in LR_MILESTONES:
                arm=arm_name(rows_,s,'lrlo')
                if have(tid,arm): continue
                new=curve_for(info['checkpoints'][str(s)]['path'],tid,arm,s,nc,nq,
                              lr=LR_LOW,l2sp=L2SP_ALPHA,regime='lrlo')
                curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
                base={r['context_size']:r['logloss'] for r in curves
                      if int(r['target'])==tid and r['arm']=='base' and int(r['repeat'])==0}
                g={r['context_size']:round(base.get(r['context_size'],np.nan)-r['logloss'],4)
                   for r in new if r['repeat']==0}
                print(f'  {arm} | gain by context: {g}',flush=True)
        except Exception as exc:
            record_failure('lrlo',tid,META[tid]['name'],exc); clean_gpu()
        finally:
            shutil.rmtree(path,ignore_errors=True); clean_gpu()
else:
    print('\nOptimiser axis not run (disabled, or insufficient budget).')

CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\nTotal: {CUR.target.nunique()} datasets, {CUR.arm.nunique()} arms, {len(CUR)} points. '
      f'Budget left: {hours_left():.2f} h.')

## 11. Results

Five readings. The third decides between the two hypotheses; the others locate
it.

In [ ]:

from scipy.stats import wilcoxon, spearmanr, binomtest

CUR=pd.read_csv(CURVE_PATH)
for c_,v_ in [('regime','standard'),('lr',LEARNING_RATE),('l2sp',0.0)]:
    if c_ not in CUR.columns: CUR[c_]=v_
KEY=['target','name','n_train','arm','ft_steps','ep_context','ep_query','episode_rows',
     'regime','lr','l2sp','context_size','requested']
AGG=(CUR.groupby(KEY,as_index=False)
        .agg(logloss=('logloss','mean'),auc=('auc','mean'),ap=('ap','mean'),
             n_context_rows=('n_context_rows','mean'),repeats=('repeat','count')))
AGG['ctx']=AGG.n_context_rows.astype(float)
write_csv(ROOT/'curves_agg.csv',AGG)

FT=AGG[AGG.arm!='base'].merge(AGG[AGG.arm=='base'][['target','context_size','logloss','auc']],
    on=['target','context_size'],suffixes=('','_base'),how='inner')
FT['gain_logloss']=FT.logloss_base-FT.logloss
FT['gain_auc']=FT.auc-FT.auc_base
FT['ratio']=FT.ctx/FT.ep_context.clip(lower=1)
write_csv(ROOT/'ft_gains.csv',FT)
LR_PROBE=FT[FT.regime=='lrlo'].copy()
FT=FT[FT.regime=='standard'].copy()      # the main grid is at a constant optimiser regime
TOP=max(FT_MILESTONES)
SIZES=sorted(FT.episode_rows.unique())
print('Episode lengths measured:',SIZES)
print('Datasets per size:',{int(s):int(FT[FT.episode_rows==s].target.nunique()) for s in SIZES})

print('\n=== 1. THE GRID, on the SUBSET COMMON to all episode lengths ===')
common=set.intersection(*[set(FT[FT.episode_rows==s].target) for s in SIZES]) if SIZES else set()
print(f'{len(common)} datasets eligible at every length.')
CO=FT[FT.target.isin(common)]
for st in FT_MILESTONES:
    sub=CO[CO.ft_steps==st]
    if sub.empty: continue
    t=(sub.pivot_table(index='context_size',columns='episode_rows',values='gain_logloss',
                       aggfunc='mean')
         .join(sub.groupby('context_size').ctx.mean().rename('rows'))
         .sort_values('rows').round(5))
    print(f'\nLog-loss gain at {st} steps - columns = rows per episode, n={len(common)}')
    display(t); write_csv(ROOT/f'grille_{st}pas.csv',t.reset_index())

print('\n=== 2. HEAD TO HEAD between episode lengths, at every context ===')
h2h=[]
for i in range(len(SIZES)):
    for j in range(i+1,len(SIZES)):
        a,b_=SIZES[i],SIZES[j]
        for cs,g in FT[FT.ft_steps==TOP].groupby('context_size'):
            p=g.pivot_table(index='target',columns='episode_rows',values='gain_logloss')
            if a not in p or b_ not in p: continue
            d=(p[b_]-p[a]).dropna()
            if len(d)<5 or d.abs().sum()==0: continue
            h2h.append(dict(context=cs,rows=round(float(g.ctx.mean()),1),
                comparison=f'{int(b_)} r. - {int(a)} r.',mean_gap=round(float(d.mean()),5),
                median_gap=round(float(d.median()),5),
                wilcoxon_p=round(float(wilcoxon(d).pvalue),5),
                wins_long=int((d>0).sum()),n=len(d)))
H2H=pd.DataFrame(h2h)
if len(H2H):
    H2H=H2H.sort_values(['comparison','rows'])
    display(H2H); write_csv(ROOT/'tete_a_tete.csv',H2H)

print('\n=== 3. A1/A2 - DOES THE RATIO EXPLAIN MORE THAN THE CONTEXT? ===')
cmp_rows=[]
for tid,g in FT[FT.ft_steps==TOP].groupby('target'):
    g=g.dropna(subset=['gain_logloss'])
    if g.episode_rows.nunique()<2 or len(g)<6: continue
    r1=spearmanr(np.log2(g.ratio),g.gain_logloss)[0]
    r2=spearmanr(np.log2(g.ctx),g.gain_logloss)[0]
    cmp_rows.append(dict(target=tid,name=g.name.iloc[0],n_train=int(g.n_train.iloc[0]),
        rho_ratio=round(float(r1),4),rho_context=round(float(r2),4),
        ratio_wins=bool(abs(r1)>abs(r2)),points=len(g)))
CMP=pd.DataFrame(cmp_rows)
if len(CMP):
    display(CMP.sort_values('rho_ratio'))
    write_csv(ROOT/'rapport_vs_contexte.csv',CMP)
    w=int(CMP.ratio_wins.sum()); n=len(CMP)
    print(f'A1: the ratio explains more than the absolute context on {w}/{n} datasets '
          f'(median |rho| {CMP.rho_ratio.abs().median():.3f} against '
          f'{CMP.rho_context.abs().median():.3f}, test des signes p={binomtest(w,n,0.5).pvalue:.5f}).')
    print('  -> collapse supported' if w>n/2 and binomtest(w,n,0.5).pvalue<0.05 else
          '  -> collapse NOT supported: say so plainly.')

print('\nA2 - gain by ratio band (context / episode context)')
EDGES=[0,0.25,0.5,1.0,2.0,4.0,8.0,np.inf]
LABS=['<=1/4','1/4-1/2','1/2-1','1-2','2-4','4-8','>8']
tr=[]
sub=FT[FT.ft_steps==TOP].dropna(subset=['gain_logloss']).copy()
sub['band']=pd.cut(sub.ratio,EDGES,labels=LABS,right=True,include_lowest=True)
for lab,g in sub.groupby('band',observed=True):
    d=g.groupby('target').gain_logloss.mean().dropna()   # one point per dataset
    if len(d)<5 or d.abs().sum()==0: continue
    tr.append(dict(band=str(lab),gain_mean=round(float(d.mean()),5),
        gain_median=round(float(d.median()),5),positive=int((d>0).sum()),n=len(d),
        wilcoxon_p=round(float(wilcoxon(d).pvalue),5)))
TR=pd.DataFrame(tr)
if len(TR): display(TR); write_csv(ROOT/'tranches_rapport.csv',TR)

print('\n=== 4. A3 - STRICT MATCHING, or \'at least as long\'? ===')
a3=[]
for cs,g in FT[FT.ft_steps==TOP].groupby('context_size'):
    c=float(g.ctx.mean()); avail=sorted(g.ep_context.unique())
    if len(avail)<2: continue
    matched=min(avail,key=lambda e:abs(np.log2(max(e,1))-np.log2(max(c,1))))
    longer=[e for e in avail if e>matched]
    if not longer: continue
    big=max(longer)
    p=g.pivot_table(index='target',columns='ep_context',values='gain_logloss')
    if matched not in p or big not in p: continue
    d=(p[big]-p[matched]).dropna()
    if len(d)<5 or d.abs().sum()==0: continue
    a3.append(dict(context=cs,rows=round(c,1),matched_episode=int(matched),
        longer_episode=int(big),gap=round(float(d.mean()),5),
        wilcoxon_p=round(float(wilcoxon(d).pvalue),5),
        wins_longer=int((d>0).sum()),n=len(d)))
A3=pd.DataFrame(a3)
if len(A3):
    A3=A3.sort_values('rows'); display(A3); write_csv(ROOT/'a3_correspondance.csv',A3)
    tot=int(A3.wins_longer.sum()); nn=int(A3.n.sum())
    print(f'A3: the longer episode wins {tot}/{nn} paired comparisons.')
    print('  -> strict matching is refuted: "at least as long" is enough.'
          if tot>nn/2 else
          '  -> strict matching is NOT refuted: that is the other hypothesis.')

print('\n=== 5. A4 - DOES THE CROSSING POINT MOVE WITH EPISODE LENGTH? ===')
cr=[]
for (tid,ep),g in FT[FT.ft_steps==TOP].groupby(['target','episode_rows']):
    g=g.dropna(subset=['gain_logloss']).sort_values('ctx')
    pos=g[g.gain_logloss>0]
    if g.empty: continue
    cr.append(dict(target=tid,name=g.name.iloc[0],episode_rows=int(ep),
        last_winner=(float(pos.ctx.max()) if len(pos) else 0.0),
        gain_at_ref=float(g[g.context_size==str(REF_CONTEXT)].gain_logloss.mean())
                    if (g.context_size==str(REF_CONTEXT)).any() else np.nan))
CR=pd.DataFrame(cr)
if len(CR):
    piv=CR.pivot_table(index='target',columns='episode_rows',values='last_winner')
    display(piv.round(0)); write_csv(ROOT/'croisements.csv',CR)
    if piv.shape[1]>=2:
        a,b_=piv.columns[0],piv.columns[-1]
        d=(np.log2(piv[b_].clip(lower=1))-np.log2(piv[a].clip(lower=1))).dropna()
        if len(d)>=5 and d.abs().sum()>0:
            print(f'A4: from {int(a)} to {int(b_)} rows per episode, the last winning context '
                  f'moves by {d.mean():+.2f} doubling(s) on average, '
                  f'{int((d>0).sum())}/{len(d)} datasets, Wilcoxon p={wilcoxon(d).pvalue:.5f}')

print('\n=== 6. A5 - DOES THE SHAPE SURVIVE A 10x SMALLER STEP? ===')
TS=pd.read_csv(TRAIN_PATH) if Path(TRAIN_PATH).exists() else pd.DataFrame()
if LR_PROBE.empty or TS.empty or 'displacement' not in TS.columns:
    print('Optimiser axis not run, or displacements missing: nothing to compare.')
else:
    DISP=TS[['target','regime','episode_rows','steps','displacement']].drop_duplicates()
    EPL=sum(LR_EPISODE); ids=sorted(LR_PROBE.target.unique())
    L=LR_PROBE.merge(DISP,left_on=['target','regime','episode_rows','ft_steps'],
                     right_on=['target','regime','episode_rows','steps'],how='left')
    S_=(FT[(FT.episode_rows==EPL)&(FT.target.isin(ids))]
          .merge(DISP,left_on=['target','regime','episode_rows','ft_steps'],
                 right_on=['target','regime','episode_rows','steps'],how='left'))
    both=pd.concat([L,S_],ignore_index=True)
    t5=(both.groupby(['regime','ft_steps'])
            .agg(mean_displacement=('displacement','mean'),
                 gain_short=('gain_logloss',lambda s:np.nan),datasets=('target','nunique'))
            .drop(columns='gain_short').round(4))
    print(f'Weight displacement by regime (episodes of {EPL} rows, n={len(ids)}):')
    display(t5)
    g5=(both.groupby(['regime','ft_steps','context_size'])
            .agg(rows=('ctx','mean'),gain_mean=('gain_logloss','mean'),
                 datasets=('target','nunique'))
            .reset_index().sort_values(['regime','ft_steps','rows']).round(5))
    display(g5); write_csv(ROOT/'axe_optimisation.csv',g5)
    # Matched-DISPLACEMENT comparison: for each low-step milestone, take the standard
    # milestone whose displacement is closest, on the same dataset.
    pairs=[]
    for (tid,cs),g in both.groupby(['target','context_size']):
        lo=g[g.regime=='lrlo'].dropna(subset=['displacement'])
        st=g[g.regime=='standard'].dropna(subset=['displacement'])
        if lo.empty or st.empty: continue
        for _,r in lo.iterrows():
            j=(st.displacement-r.displacement).abs().idxmin()
            pairs.append(dict(target=tid,context_size=cs,rows=float(r.ctx),
                steps_low_lr=int(r.ft_steps),disp_low_lr=float(r.displacement),
                steps_standard=int(st.loc[j,'ft_steps']),disp_standard=float(st.loc[j,'displacement']),
                gain_low_lr=float(r.gain_logloss),gain_standard=float(st.loc[j,'gain_logloss'])))
    PR=pd.DataFrame(pairs)
    if len(PR):
        PR['gap']=PR.gain_low_lr-PR.gain_standard
        write_csv(ROOT/'axe_optimisation_apparie.csv',PR)
        t6=(PR.groupby('context_size')
              .agg(rows=('rows','mean'),gain_low_lr=('gain_low_lr','mean'),
                   gain_standard=('gain_standard','mean'),gap=('gap','mean'),
                   n=('target','nunique')).sort_values('rows').round(5))
        print('\nAt matched weight displacement - positive = the small step does better')
        display(t6); write_csv(ROOT/'axe_optimisation_table.csv',t6.reset_index())
        d=PR.groupby('target').gap.mean().dropna()
        if len(d)>=3 and d.abs().sum()>0:
            print(f'A5: mean gap {d.mean():+.5f} over {len(d)} datasets '
                  f'({int((d>0).sum())} in favour of the small step).')
        gr=PR[PR.rows>=REF_CONTEXT]
        if len(gr):
            print(f'  At context >= {REF_CONTEXT}: small step {gr.gain_low_lr.mean():+.5f}, '
                  f'standard {gr.gain_standard.mean():+.5f}. '
                  +('The damage persists: the law does not come from the step size.'
                    if gr.gain_low_lr.mean()<0 else
                    'The damage disappears: two mechanisms, to be disentangled.'))
    else:
        print('No pair at comparable displacement: the standard arm of these datasets '
              'comes from the import and has no measured displacement. Re-run the optimiser '
              'axis on datasets trained in this session.')

## 12. Figures

Figure 2 is the one that matters: if the three curves fall on top of one another
when plotted against the ratio, the claim is made in a single image.

In [ ]:

RAMP=['#8FB4DC','#4B7FBB','#1F4E79']; MARKS=['o','s','^']
ORANGE='#D1793A'; GREY='#6B7280'
plt.rcParams.update({'figure.dpi':160,'savefig.dpi':300,'font.size':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.edgecolor':'#9CA3AF','axes.labelcolor':'#374151',
                     'xtick.color':GREY,'ytick.color':GREY,'text.color':'#1F2937'})
def col(i): return RAMP[min(i,len(RAMP)-1)]
def mark(i): return MARKS[min(i,len(MARKS)-1)]

S=FT[(FT.ft_steps==TOP)&(FT.target.isin(common))] if common else FT[FT.ft_steps==TOP]

# --- Figure 1: gain against context, one curve per episode length
if len(S):
    fig,ax=plt.subplots(figsize=(5.8,4.2)); ax.set_xscale('log')
    ax.axhline(0,color=ORANGE,lw=2.0,zorder=2)
    for i,ep in enumerate(SIZES):
        g=(S[(S.episode_rows==ep)&(S.context_size!='full')]
             .groupby('context_size').agg(x=('ctx','mean'),y=('gain_logloss','mean'),
             sd=('gain_logloss','std'),n=('target','nunique')).sort_values('x'))
        if len(g)<2: continue
        ax.errorbar(g.x,g.y,yerr=g.sd/np.sqrt(g.n.clip(lower=1)),color=col(i),lw=1.9,
                    marker=mark(i),ms=6,capsize=2.5,zorder=3,label=f'episodes of {int(ep)} rows')
    ax.set_xlabel('Rows in the inference context (log scale)')
    ax.set_ylabel('Log-loss gain')
    ax.set_title(f'Does the crossing point move? (n={S.target.nunique()}, {TOP} steps)',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig1_par_episode.png',bbox_inches='tight'); plt.show()

# --- Figure 2: the collapse against the ratio
if len(S) and S.episode_rows.nunique()>=2:
    fig,ax=plt.subplots(figsize=(5.8,4.2)); ax.set_xscale('log',base=2)
    ax.axhline(0,color=ORANGE,lw=2.0,zorder=2)
    ax.axvline(1,color='#374151',lw=1.1,ls='--',zorder=1)
    for i,ep in enumerate(SIZES):
        g=S[S.episode_rows==ep].dropna(subset=['gain_logloss'])
        if g.empty: continue
        gb=(g.assign(b=np.round(np.log2(g.ratio)*2)/2)
              .groupby('b').agg(x=('ratio','median'),y=('gain_logloss','mean'),
              sd=('gain_logloss','std'),n=('target','nunique')).sort_values('x'))
        gb=gb[gb.n>=3]           # a mean over 1 or 2 datasets is not a mean
        if not len(gb): continue
        ax.errorbar(gb.x,gb.y,yerr=gb.sd/np.sqrt(gb.n.clip(lower=1)),color=col(i),lw=1.7,
                    marker=mark(i),ms=5.5,capsize=2,alpha=.95,zorder=3,
                    label=f'episodes of {int(ep)} rows')
    y0,y1=ax.get_ylim(); ax.set_ylim(y0-0.22*(y1-y0),y1)
    ax.annotate('deployed shorter\nthan trained',xy=(1,y0-0.19*(y1-y0)),
                xytext=(-8,0),textcoords='offset points',ha='right',va='bottom',
                fontsize=7.5,color=GREY)
    ax.annotate('extrapolating\nbeyond',xy=(1,y0-0.19*(y1-y0)),xytext=(8,0),
                textcoords='offset points',ha='left',va='bottom',fontsize=7.5,color=GREY)
    ax.set_xlabel('Ratio  inference context / training context')
    ax.set_ylabel('Log-loss gain')
    ax.set_title('A single curve, if the ratio is the right variable',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig2_effondrement.png',bbox_inches='tight'); plt.show()

# --- Figure 3: how the crossing point moves, dataset by dataset
if len(CR) and CR.episode_rows.nunique()>=2:
    piv=CR.pivot_table(index='target',columns='episode_rows',values='last_winner')
    fig,ax=plt.subplots(figsize=(5.4,3.9)); ax.set_yscale('log')
    for tid,row in piv.iterrows():
        v=row.dropna()
        if len(v)<2: continue
        ax.plot([int(c) for c in v.index],v.clip(lower=16).values,lw=1.0,color=RAMP[1],
                alpha=.35,marker='o',ms=3,mew=0,zorder=2)
    m=piv.median()
    ax.plot([int(c) for c in m.index],m.clip(lower=16).values,lw=2.4,color=RAMP[2],
            marker='s',ms=7,zorder=4,label='median')
    ax.set_xscale('log',base=2)
    ax.set_xlabel('Rows per training episode')
    ax.set_ylabel('Last context where fine-tuning still wins')
    ax.set_title('Training longer pushes the limit out',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig3_croisement.png',bbox_inches='tight'); plt.show()

write_json(ROOT/'RUN_COMPLETE.json',dict(run_hash=RUN_HASH,
    datasets=int(CUR.target.nunique()),arms=sorted(CUR.arm.unique().tolist()),
    episode_sizes=[int(s) for s in SIZES],common_subset=sorted(int(t) for t in common),
    points=len(CUR),excluded=FAILURES,hours_left=round(hours_left(),2),
    note='development study; OpenML outer fold 0 never read'))
print('\nFiles in',ROOT)
for f in sorted(ROOT.glob('*.csv'))+sorted(ROOT.glob('*.png'))+sorted(ROOT.glob('*.json')):
    print('  ',f.name)

## How to read this run

**The first thing to check** is the step budget, printed in the results section.
The whole point of this notebook is that every episode length sees the same
number of supervised query rows, so if the realised step counts differ across
lengths the control has not done its job and the comparison is still confounded.

**Then the gap between the longest and the shortest episode at large contexts.**
In the previous run that gap could be attributed either to episode length or to
the sixteen-fold difference in supervised targets. Here supervision is equal by
construction, so whatever survives is the length.

**If the effect disappears**, it was the supervision, and that is the result. It
would mean the earlier grid measured how many targets the model saw rather than
how long its episodes were, and the write-up would have to say so plainly rather
than reach for an explanation.

**What this run cannot say.** It re-uses only the arms of the previous run whose
query count matches the control; anything else is rejected at import. That keeps
the comparison clean but reduces the sample, and the surviving count is printed
next to every number.